# NGD Colab Optimized Pipeline (L4/G4)

Technical goals:
- high-quality training defaults that remain stable on L4/G4 Colab GPUs,
- resilient checkpoint backup to Drive for runtime disconnect recovery,
- automatic candidate submission sweep with latency guard,
- exactly two final deliverables per run:
  1. selected submission zip (`run.py` + `model.onnx`),
  2. audit zip (weights + metadata + checksums).

This notebook is self-contained by default (`CODE_SOURCE='embedded'`) and does not require Git checkout in Colab.



In [ ]:
# ---- 0) CONFIG ----
from pathlib import Path

CFG = {
    # Code source (embedded|upload_zip|git|drive)
    'CODE_SOURCE': 'embedded',
    'SOURCE_ZIP_IN_DRIVE': '',  # optional: /content/drive/MyDrive/.../norgesgruppen-data-src.zip
    'GIT_URL': 'https://github.com/<org>/<repo>.git',
    'GIT_REF': 'main',
    'GIT_TOKEN': '',  # optional; leave empty to paste token securely at runtime prompt
    'DRIVE_CODE_PATH': '/content/drive/MyDrive/NMIAI26/code/norgesgruppen-data',
    'WORKDIR': '/content/nmiai_ngd',

    # Durable storage
    'DRIVE_ROOT': '/content/drive/MyDrive/NMIAI26/ngd',
    'DELIVERABLES_DIR_IN_DRIVE': '/content/drive/MyDrive/NMIAI26/ngd/deliverables',
    'DATASET_ZIP_IN_DRIVE': '/content/drive/MyDrive/NMIAI26/datasets/NM_NGD_coco_dataset.zip',

    # Training (optimized default for L4/G4 quality/runtime balance)
    'MODEL': 'yolo11x.pt',
    'EPOCHS': 220,
    'IMGSZ': 1280,
    'BATCH': 4,
    'PATIENCE': 100,
    'WORKERS': 8,
    'VAL_RATIO': 0.12,
    'SEED': 42,
    'MOSAIC': 0.7,
    'CLOSE_MOSAIC': 25,
    'DEVICE': 'auto',
    'RESUME_FROM_BACKUP': True,
    'RUN_NAME': 'ngd_y11x_1280_e220_l4g4_opt_s42',

    # Runtime-safe candidate selection under 360s evaluator budget.
    'EVAL_TIME_LIMIT_SEC': 360,
    'EST_EVAL_IMAGES': 400,
    'SAFETY_FACTOR': 0.80,
    'BENCHMARK_SAMPLE_IMAGES': 40,
    'BENCHMARK_TIMEOUT_SEC': 240,

    # Ordered by quality priority (top first). Notebook selects first safe profile.
    'SUBMISSION_PROFILES': [
        {'name': 'q1_x1024_c025_i060', 'export_imgsz': 1024, 'conf': 0.025, 'iou': 0.60, 'topk': 1500, 'max_det': 260},
        {'name': 'q2_x1024_c030_i060', 'export_imgsz': 1024, 'conf': 0.030, 'iou': 0.60, 'topk': 1300, 'max_det': 220},
        {'name': 'q3_x1024_c035_i060', 'export_imgsz': 1024, 'conf': 0.035, 'iou': 0.60, 'topk': 1200, 'max_det': 210},
        {'name': 'q4_x1024_c040_i060', 'export_imgsz': 1024, 'conf': 0.040, 'iou': 0.60, 'topk': 1100, 'max_det': 200},
        {'name': 's1_x0960_c035_i060', 'export_imgsz': 960, 'conf': 0.035, 'iou': 0.60, 'topk': 1050, 'max_det': 190},
        {'name': 's2_x0960_c045_i060', 'export_imgsz': 960, 'conf': 0.045, 'iou': 0.60, 'topk': 900, 'max_det': 170},
    ],
}

# Export ONNX only for sizes needed by profile list.
CFG['EXPORT_IMGSZ'] = sorted({int(p['export_imgsz']) for p in CFG['SUBMISSION_PROFILES']}, reverse=True)

WORKDIR = Path(CFG['WORKDIR'])
REPO_DIR = WORKDIR
RUN_DIR = REPO_DIR / 'runs' / CFG['RUN_NAME']
DRIVE_ROOT = Path(CFG['DRIVE_ROOT'])
BACKUP_DIR = DRIVE_ROOT / 'checkpoints' / CFG['RUN_NAME']
AUDIT_DIR = DRIVE_ROOT / 'audit'
SUB_DIR = DRIVE_ROOT / 'submissions' / CFG['RUN_NAME']
DELIVERABLES_DIR = Path(CFG['DELIVERABLES_DIR_IN_DRIVE']) / CFG['RUN_NAME']
print(CFG)



In [ ]:
# ---- 1) Install dependencies ----
%pip -q install -U pip
%pip -q install ultralytics onnxruntime onnx numpy pillow pyyaml


In [ ]:
# ---- 2) Mount Drive ----
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
BACKUP_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
SUB_DIR.mkdir(parents=True, exist_ok=True)
DELIVERABLES_DIR.mkdir(parents=True, exist_ok=True)
print('Drive ready:', DRIVE_ROOT)


In [ ]:
# ---- 3) Bootstrap code in /content ----
import os, shutil, subprocess, getpass, zipfile

if WORKDIR.exists():
    shutil.rmtree(WORKDIR)
WORKDIR.parent.mkdir(parents=True, exist_ok=True)

ignore = shutil.ignore_patterns(
    '.git',
    'node_modules',
    'artifacts',
    'data/raw',
    '__pycache__',
    '.pytest_cache',
    '.DS_Store',
    '*.pt',
    '*.onnx',
    '*.onnx.data',
    '*.zip',
)

source_mode = str(CFG.get('CODE_SOURCE', 'embedded')).strip().lower()
if source_mode == 'embedded':
    embedded_files = {
    'src/data/prepare_yolo_dataset.py': '#!/usr/bin/env python3\n"""Prepare NM_NGD COCO dataset for Ultralytics YOLO training.\n\nOutputs:\n- training/yolo_dataset/images/{train,val}/*.jpg\n- training/yolo_dataset/labels/{train,val}/*.txt\n- training/yolo_dataset/data.yaml\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport random\nimport shutil\nimport zipfile\nfrom collections import defaultdict\nfrom pathlib import Path\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\n        "--zip-path",\n        type=Path,\n        default=Path("../NM_NGD_coco_dataset.zip"),\n        help="Path to NM_NGD_coco_dataset.zip",\n    )\n    parser.add_argument("--out-dir", type=Path, default=Path("./yolo_dataset"))\n    parser.add_argument("--val-ratio", type=float, default=0.12)\n    parser.add_argument("--seed", type=int, default=42)\n    return parser.parse_args()\n\n\ndef coco_to_yolo_bbox(bbox: list[float], width: int, height: int) -> tuple[float, float, float, float]:\n    x, y, w, h = bbox\n    cx = x + (w / 2.0)\n    cy = y + (h / 2.0)\n    return cx / width, cy / height, w / width, h / height\n\n\ndef main() -> None:\n    args = parse_args()\n    out_dir = args.out_dir.resolve()\n    zip_path = args.zip_path.resolve()\n\n    if out_dir.exists():\n        shutil.rmtree(out_dir)\n    (out_dir / "images" / "train").mkdir(parents=True, exist_ok=True)\n    (out_dir / "images" / "val").mkdir(parents=True, exist_ok=True)\n    (out_dir / "labels" / "train").mkdir(parents=True, exist_ok=True)\n    (out_dir / "labels" / "val").mkdir(parents=True, exist_ok=True)\n\n    with zipfile.ZipFile(zip_path) as zf:\n        with zf.open("train/annotations.json") as f:\n            coco = json.load(f)\n\n        categories = sorted(coco["categories"], key=lambda c: int(c["id"]))\n        cat_ids = [int(c["id"]) for c in categories]\n        if cat_ids != list(range(len(cat_ids))):\n            raise RuntimeError("Category IDs must be contiguous 0..N-1 for YOLO.")\n\n        images = sorted(coco["images"], key=lambda im: int(im["id"]))\n        anns_by_image: dict[int, list[dict]] = defaultdict(list)\n        for ann in coco["annotations"]:\n            anns_by_image[int(ann["image_id"])].append(ann)\n\n        rnd = random.Random(args.seed)\n        image_ids = [int(im["id"]) for im in images]\n        rnd.shuffle(image_ids)\n        val_count = max(1, int(len(image_ids) * args.val_ratio))\n        val_ids = set(image_ids[:val_count])\n\n        # copy images and emit labels\n        id_to_meta = {int(im["id"]): im for im in images}\n        for image_id in image_ids:\n            meta = id_to_meta[image_id]\n            file_name = meta["file_name"]\n            width = int(meta["width"])\n            height = int(meta["height"])\n            split = "val" if image_id in val_ids else "train"\n\n            src_name = f"train/images/{file_name}"\n            dst_image = out_dir / "images" / split / file_name\n            with zf.open(src_name) as src, open(dst_image, "wb") as dst:\n                shutil.copyfileobj(src, dst)\n\n            dst_label = out_dir / "labels" / split / (Path(file_name).stem + ".txt")\n            lines: list[str] = []\n            for ann in anns_by_image.get(image_id, []):\n                cat = int(ann["category_id"])\n                x_c, y_c, w_n, h_n = coco_to_yolo_bbox(ann["bbox"], width, height)\n                # Clamp to valid normalized range.\n                x_c = min(max(x_c, 0.0), 1.0)\n                y_c = min(max(y_c, 0.0), 1.0)\n                w_n = min(max(w_n, 0.0), 1.0)\n                h_n = min(max(h_n, 0.0), 1.0)\n                if w_n <= 0.0 or h_n <= 0.0:\n                    continue\n                lines.append(f"{cat} {x_c:.6f} {y_c:.6f} {w_n:.6f} {h_n:.6f}")\n            dst_label.write_text("\\n".join(lines))\n\n    # data.yaml\n    names = [c["name"] for c in categories]\n    yaml_lines = [\n        f"path: {out_dir.as_posix()}",\n        "train: images/train",\n        "val: images/val",\n        f"nc: {len(names)}",\n        "names:",\n    ]\n    for idx, name in enumerate(names):\n        # Quote safely for YAML.\n        safe = name.replace(\'"\', "\'")\n        yaml_lines.append(f\'  {idx}: "{safe}"\')\n    (out_dir / "data.yaml").write_text("\\n".join(yaml_lines) + "\\n")\n\n    print(f"Prepared dataset at {out_dir}")\n    print(f"Images: {len(images)} (train={len(images)-len(val_ids)}, val={len(val_ids)})")\n    print(f"Categories: {len(names)}")\n\n\nif __name__ == "__main__":\n    main()\n',
    'src/train/train_yolov8.py': '#!/usr/bin/env python3\n"""Train YOLOv8 for NM_NGD on local machine (MPS/CUDA/CPU)."""\n\nfrom __future__ import annotations\n\nimport argparse\nfrom pathlib import Path\n\nimport torch\nfrom ultralytics import YOLO\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--data", type=Path, default=Path("./yolo_dataset/data.yaml"))\n    parser.add_argument("--model", type=str, default="yolov8n.pt")\n    parser.add_argument("--epochs", type=int, default=40)\n    parser.add_argument("--imgsz", type=int, default=960)\n    parser.add_argument("--batch", type=int, default=8)\n    parser.add_argument("--patience", type=int, default=12)\n    parser.add_argument("--workers", type=int, default=4)\n    parser.add_argument("--name", type=str, default="ngd_yolov8n_mps")\n    parser.add_argument("--seed", type=int, default=42)\n    parser.add_argument("--no-val", action="store_true", help="Disable built-in val loop.")\n    parser.add_argument(\n        "--device",\n        type=str,\n        default="auto",\n        help="Device override: auto|mps|cpu|0 (CUDA index).",\n    )\n    parser.add_argument(\n        "--resume",\n        action="store_true",\n        help="Resume from the checkpoint given by --model (typically runs/<name>/weights/last.pt).",\n    )\n    parser.add_argument("--mosaic", type=float, default=1.0, help="Mosaic augmentation probability.")\n    parser.add_argument("--close-mosaic", type=int, default=10, help="Disable mosaic in final N epochs.")\n    parser.add_argument(\n        "--deterministic",\n        action="store_true",\n        help="Enable deterministic training mode (may be less stable on MPS).",\n    )\n    return parser.parse_args()\n\n\ndef pick_device() -> str:\n    if torch.cuda.is_available():\n        return "0"\n    if torch.backends.mps.is_available():\n        return "mps"\n    return "cpu"\n\n\ndef main() -> None:\n    args = parse_args()\n    device = pick_device() if args.device == "auto" else args.device\n    print(f"Using device: {device}")\n    use_amp = device != "mps"\n    print(f"AMP enabled: {use_amp}")\n\n    # Compatibility for PyTorch 2.6+ default weights_only=True with legacy Ultralytics checkpoints.\n    original_torch_load = torch.load\n\n    def patched_torch_load(*a, **kw):\n        kw.setdefault("weights_only", False)\n        return original_torch_load(*a, **kw)\n\n    torch.load = patched_torch_load\n    try:\n        model = YOLO(args.model)\n    finally:\n        torch.load = original_torch_load\n\n    model.train(\n        data=str(args.data.resolve()),\n        epochs=args.epochs,\n        imgsz=args.imgsz,\n        batch=args.batch,\n        patience=args.patience,\n        workers=args.workers,\n        project=str(Path("./runs").resolve()),\n        name=args.name,\n        seed=args.seed,\n        device=device,\n        pretrained=True,\n        exist_ok=True,\n        close_mosaic=args.close_mosaic,\n        mosaic=args.mosaic,\n        lr0=0.005,\n        lrf=0.05,\n        optimizer="AdamW",\n        amp=use_amp,\n        val=not args.no_val,\n        resume=args.resume,\n        deterministic=args.deterministic,\n    )\n\n\nif __name__ == "__main__":\n    main()\n',
    'src/inference/run.py': "#!/usr/bin/env python3\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport re\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\nimport onnxruntime as ort\nfrom PIL import Image\n\nIMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png'}\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser()\n    parser.add_argument('--input', required=True, type=Path)\n    parser.add_argument('--output', required=True, type=Path)\n    return parser.parse_args()\n\n\ndef parse_image_id(path: Path) -> int:\n    match = re.search(r'img_(\\d+)$', path.stem, flags=re.IGNORECASE)\n    if match:\n        return int(match.group(1))\n    digits = ''.join(ch for ch in path.stem if ch.isdigit())\n    return int(digits) if digits else 0\n\n\ndef iter_images(input_dir: Path) -> list[Path]:\n    return sorted(p for p in input_dir.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)\n\n\ndef clamp_bbox(x: float, y: float, w: float, h: float, width: int, height: int) -> list[float]:\n    x = max(0.0, min(x, float(width - 1)))\n    y = max(0.0, min(y, float(height - 1)))\n    w = max(1.0, min(w, float(width) - x))\n    h = max(1.0, min(h, float(height) - y))\n    return [round(x, 2), round(y, 2), round(w, 2), round(h, 2)]\n\n\ndef fallback_predictions(images: list[Path]) -> list[dict[str, Any]]:\n    predictions: list[dict[str, Any]] = []\n    for image_path in images:\n        image_id = parse_image_id(image_path)\n        with Image.open(image_path) as image:\n            width, height = image.size\n\n        band_h = max(20.0, height * 0.22)\n        y_starts = [height * 0.18, height * 0.42, height * 0.66]\n        for idx, y in enumerate(y_starts):\n            bbox = clamp_bbox(0.0, y, float(width), band_h, width, height)\n            predictions.append(\n                {\n                    'image_id': image_id,\n                    'category_id': 0,\n                    'bbox': bbox,\n                    'score': round(0.25 - idx * 0.02, 3),\n                }\n            )\n    return predictions\n\n\ndef _nms(boxes_xyxy: np.ndarray, scores: np.ndarray, iou_thr: float, max_det: int) -> np.ndarray:\n    if boxes_xyxy.size == 0:\n        return np.empty((0,), dtype=np.int64)\n\n    x1 = boxes_xyxy[:, 0]\n    y1 = boxes_xyxy[:, 1]\n    x2 = boxes_xyxy[:, 2]\n    y2 = boxes_xyxy[:, 3]\n    areas = np.maximum(0.0, x2 - x1) * np.maximum(0.0, y2 - y1)\n    order = scores.argsort()[::-1]\n\n    keep: list[int] = []\n    while order.size > 0 and len(keep) < max_det:\n        i = int(order[0])\n        keep.append(i)\n        if order.size == 1:\n            break\n        rest = order[1:]\n\n        xx1 = np.maximum(x1[i], x1[rest])\n        yy1 = np.maximum(y1[i], y1[rest])\n        xx2 = np.minimum(x2[i], x2[rest])\n        yy2 = np.minimum(y2[i], y2[rest])\n\n        w = np.maximum(0.0, xx2 - xx1)\n        h = np.maximum(0.0, yy2 - yy1)\n        inter = w * h\n        union = areas[i] + areas[rest] - inter + 1e-9\n        iou = inter / union\n\n        order = rest[iou <= iou_thr]\n\n    return np.asarray(keep, dtype=np.int64)\n\n\ndef _preprocess(image: Image.Image, input_hw: tuple[int, int]) -> np.ndarray:\n    in_h, in_w = input_hw\n    arr = np.asarray(image.resize((in_w, in_h)), dtype=np.float32) / 255.0\n    return np.transpose(arr, (2, 0, 1))[None, ...]\n\n\ndef onnx_predictions(images: list[Path]) -> list[dict[str, Any]]:\n    model_path = Path('model.onnx')\n    if not model_path.exists():\n        raise FileNotFoundError('model.onnx not found next to run.py')\n\n    providers = ['CPUExecutionProvider']\n    available = ort.get_available_providers()\n    if 'CUDAExecutionProvider' in available:\n        providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']\n\n    session = ort.InferenceSession(str(model_path), providers=providers)\n    input_name = session.get_inputs()[0].name\n    _, _, in_h, in_w = session.get_inputs()[0].shape\n\n    conf_thr = 0.03\n    iou_thr = 0.60\n    max_det = 300\n\n    predictions: list[dict[str, Any]] = []\n    for image_path in images:\n        image_id = parse_image_id(image_path)\n        with Image.open(image_path) as image:\n            image = image.convert('RGB')\n            width, height = image.size\n            inp = _preprocess(image, (int(in_h), int(in_w)))\n\n        out = session.run(None, {input_name: inp})[0]\n        pred = out[0].T\n\n        boxes_xywh = pred[:, :4]\n        cls_scores = pred[:, 4:]\n        cls_idx = cls_scores.argmax(axis=1).astype(np.int64)\n        conf = cls_scores.max(axis=1)\n        keep_conf = conf >= conf_thr\n        if not np.any(keep_conf):\n            continue\n\n        boxes_xywh = boxes_xywh[keep_conf]\n        conf = conf[keep_conf]\n        cls_idx = cls_idx[keep_conf]\n\n        topk = min(2000, conf.shape[0])\n        order = np.argpartition(conf, -topk)[-topk:]\n        boxes_xywh = boxes_xywh[order]\n        conf = conf[order]\n        cls_idx = cls_idx[order]\n\n        cx = boxes_xywh[:, 0]\n        cy = boxes_xywh[:, 1]\n        bw = boxes_xywh[:, 2]\n        bh = boxes_xywh[:, 3]\n        x1 = cx - bw / 2.0\n        y1 = cy - bh / 2.0\n        x2 = cx + bw / 2.0\n        y2 = cy + bh / 2.0\n        boxes_xyxy = np.stack([x1, y1, x2, y2], axis=1)\n\n        keep_nms = _nms(boxes_xyxy, conf, iou_thr=iou_thr, max_det=max_det)\n        if keep_nms.size == 0:\n            continue\n\n        sx = width / float(in_w)\n        sy = height / float(in_h)\n\n        for i in keep_nms.tolist():\n            x1i, y1i, x2i, y2i = boxes_xyxy[i]\n            x = x1i * sx\n            y = y1i * sy\n            w = (x2i - x1i) * sx\n            h = (y2i - y1i) * sy\n            bbox = clamp_bbox(float(x), float(y), float(w), float(h), width, height)\n            predictions.append(\n                {\n                    'image_id': image_id,\n                    'category_id': int(cls_idx[i]),\n                    'bbox': bbox,\n                    'score': round(float(conf[i]), 4),\n                }\n            )\n\n    return predictions\n\n\ndef main() -> None:\n    args = parse_args()\n    images = iter_images(args.input)\n    if not images:\n        args.output.parent.mkdir(parents=True, exist_ok=True)\n        args.output.write_text('[]')\n        return\n\n    try:\n        predictions = onnx_predictions(images)\n        if not predictions:\n            predictions = fallback_predictions(images)\n    except Exception:\n        predictions = fallback_predictions(images)\n\n    args.output.parent.mkdir(parents=True, exist_ok=True)\n    args.output.write_text(json.dumps(predictions, separators=(',', ':')))\n\n\nif __name__ == '__main__':\n    main()\n",
    'scripts/watch_checkpoint_sync.py': '#!/usr/bin/env python3\n"""Continuously mirror critical training artifacts to durable storage."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport os\nimport shutil\nimport time\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nDEFAULT_FILES = [\n    "weights/best.pt",\n    "weights/last.pt",\n    "args.yaml",\n    "results.csv",\n    "results.png",\n    "labels.jpg",\n    "confusion_matrix.png",\n    "confusion_matrix_normalized.png",\n]\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(\n        description="Copy updated training artifacts from run dir to a backup dir."\n    )\n    parser.add_argument("--run-dir", required=True, type=Path, help="Ultralytics run directory.")\n    parser.add_argument("--backup-dir", required=True, type=Path, help="Durable destination directory.")\n    parser.add_argument(\n        "--files",\n        nargs="+",\n        default=DEFAULT_FILES,\n        help="Relative file paths inside run dir to mirror.",\n    )\n    parser.add_argument("--watch", action="store_true", help="Keep syncing until stopped.")\n    parser.add_argument(\n        "--wait-for-run-dir",\n        action="store_true",\n        help="In watch mode, wait until --run-dir appears instead of exiting.",\n    )\n    parser.add_argument("--interval", type=int, default=90, help="Seconds between sync passes in watch mode.")\n    parser.add_argument(\n        "--max-hours",\n        type=float,\n        default=0.0,\n        help="Optional safety stop. 0 means no time limit.",\n    )\n    return parser.parse_args()\n\n\ndef utc_now() -> str:\n    return datetime.now(timezone.utc).isoformat()\n\n\ndef file_sig(path: Path) -> tuple[int, int]:\n    stat = path.stat()\n    return (stat.st_size, stat.st_mtime_ns)\n\n\ndef copy_atomic(src: Path, dst: Path) -> None:\n    dst.parent.mkdir(parents=True, exist_ok=True)\n    tmp = dst.with_name(f"{dst.name}.tmp")\n    shutil.copy2(src, tmp)\n    os.replace(tmp, dst)\n\n\ndef write_sync_state(backup_dir: Path, run_dir: Path, state: dict[str, tuple[int, int]]) -> None:\n    payload = {\n        "updated_utc": utc_now(),\n        "run_dir": str(run_dir),\n        "tracked_files": [\n            {"relative_path": rel, "size_bytes": sig[0], "mtime_ns": sig[1]}\n            for rel, sig in sorted(state.items())\n        ],\n    }\n    (backup_dir / "sync_state.json").write_text(json.dumps(payload, indent=2))\n\n\ndef sync_once(\n    run_dir: Path, backup_dir: Path, rel_files: list[str], state: dict[str, tuple[int, int]]\n) -> list[str]:\n    copied: list[str] = []\n    for rel in rel_files:\n        src = run_dir / rel\n        if not src.exists() or not src.is_file():\n            continue\n        sig = file_sig(src)\n        if state.get(rel) == sig:\n            continue\n        dst = backup_dir / rel\n        copy_atomic(src, dst)\n        state[rel] = sig\n        copied.append(rel)\n    if copied:\n        write_sync_state(backup_dir=backup_dir, run_dir=run_dir, state=state)\n    return copied\n\n\ndef main() -> int:\n    args = parse_args()\n    rel_files = [str(Path(p)) for p in args.files]\n\n    backup_dir = args.backup_dir.resolve()\n    backup_dir.mkdir(parents=True, exist_ok=True)\n    state: dict[str, tuple[int, int]] = {}\n\n    start = time.monotonic()\n    pass_idx = 0\n\n    while True:\n        pass_idx += 1\n        run_dir = args.run_dir.resolve()\n        elapsed_hours = (time.monotonic() - start) / 3600.0\n\n        if not run_dir.exists():\n            if args.max_hours > 0 and elapsed_hours >= args.max_hours:\n                print(f"[{utc_now()}] max runtime reached ({args.max_hours}h), exiting", flush=True)\n                break\n            if args.watch and args.wait_for_run_dir:\n                print(\n                    f"[{utc_now()}] pass={pass_idx} waiting for run dir: {run_dir}",\n                    flush=True,\n                )\n                time.sleep(max(1, args.interval))\n                continue\n            raise FileNotFoundError(f"run dir not found: {run_dir}")\n\n        copied = sync_once(run_dir=run_dir, backup_dir=backup_dir, rel_files=rel_files, state=state)\n        if copied:\n            print(f"[{utc_now()}] pass={pass_idx} copied={len(copied)} files: {copied}", flush=True)\n        else:\n            print(f"[{utc_now()}] pass={pass_idx} no changes", flush=True)\n\n        if not args.watch:\n            break\n\n        if args.max_hours > 0 and elapsed_hours >= args.max_hours:\n            print(f"[{utc_now()}] max runtime reached ({args.max_hours}h), exiting", flush=True)\n            break\n\n        time.sleep(max(1, args.interval))\n\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n',
    'scripts/archive_training_run.py': '#!/usr/bin/env python3\n"""Create a reproducibility/audit bundle for one training run."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport platform\nimport shutil\nimport subprocess\nimport sys\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nDEFAULT_RUN_FILES = [\n    "weights/best.pt",\n    "weights/last.pt",\n    "args.yaml",\n    "results.csv",\n    "results.png",\n    "labels.jpg",\n    "confusion_matrix.png",\n    "confusion_matrix_normalized.png",\n]\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(description="Bundle run artifacts and metadata for accountability.")\n    parser.add_argument("--run-dir", required=True, type=Path, help="Ultralytics run directory.")\n    parser.add_argument("--output-dir", required=True, type=Path, help="Destination directory for bundles.")\n    parser.add_argument("--run-name", type=str, default=None, help="Optional name override.")\n    parser.add_argument("--dataset-path", type=Path, default=None, help="Optional dataset zip/file to fingerprint.")\n    parser.add_argument("--onnx-path", type=Path, default=None, help="Optional ONNX file to include.")\n    parser.add_argument(\n        "--extra-file",\n        type=Path,\n        action="append",\n        default=[],\n        help="Additional file(s) to include. Can be repeated.",\n    )\n    parser.add_argument("--note", type=str, default="", help="Optional run note (stored in manifest).")\n    parser.add_argument("--skip-pip-freeze", action="store_true", help="Skip requirements.lock.txt export.")\n    parser.add_argument("--skip-dataset-hash", action="store_true", help="Skip dataset checksum computation.")\n    parser.add_argument("--no-zip", action="store_true", help="Do not zip the bundle directory.")\n    return parser.parse_args()\n\n\ndef sha256_file(path: Path) -> str:\n    hasher = hashlib.sha256()\n    with path.open("rb") as f:\n        for chunk in iter(lambda: f.read(1024 * 1024), b""):\n            hasher.update(chunk)\n    return hasher.hexdigest()\n\n\ndef utc_now() -> str:\n    return datetime.now(timezone.utc).isoformat()\n\n\ndef copy_file(src: Path, dst: Path) -> None:\n    dst.parent.mkdir(parents=True, exist_ok=True)\n    shutil.copy2(src, dst)\n\n\ndef main() -> int:\n    args = parse_args()\n    run_dir = args.run_dir.resolve()\n    if not run_dir.exists():\n        raise FileNotFoundError(f"run dir not found: {run_dir}")\n\n    run_name = args.run_name or run_dir.name\n    stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")\n    bundle_dir = args.output_dir.resolve() / f"{run_name}_{stamp}"\n    bundle_dir.mkdir(parents=True, exist_ok=False)\n\n    copied_files: list[str] = []\n    checksums: dict[str, str] = {}\n\n    for rel in DEFAULT_RUN_FILES:\n        src = run_dir / rel\n        if not src.exists() or not src.is_file():\n            continue\n        dst = bundle_dir / rel\n        copy_file(src, dst)\n        rel_dst = str(dst.relative_to(bundle_dir))\n        copied_files.append(rel_dst)\n        checksums[rel_dst] = sha256_file(dst)\n\n    if args.onnx_path is not None and args.onnx_path.exists() and args.onnx_path.is_file():\n        dst = bundle_dir / "model.onnx"\n        copy_file(args.onnx_path.resolve(), dst)\n        copied_files.append("model.onnx")\n        checksums["model.onnx"] = sha256_file(dst)\n\n    for extra in args.extra_file:\n        if not extra.exists() or not extra.is_file():\n            continue\n        dst = bundle_dir / "extras" / extra.name\n        copy_file(extra.resolve(), dst)\n        rel_dst = str(dst.relative_to(bundle_dir))\n        copied_files.append(rel_dst)\n        checksums[rel_dst] = sha256_file(dst)\n\n    dataset_info: dict[str, str | int | None] = {\n        "path": None,\n        "size_bytes": None,\n        "sha256": None,\n    }\n    if args.dataset_path is not None and args.dataset_path.exists() and args.dataset_path.is_file():\n        dataset_path = args.dataset_path.resolve()\n        dataset_info["path"] = str(dataset_path)\n        dataset_info["size_bytes"] = dataset_path.stat().st_size\n        if not args.skip_dataset_hash:\n            dataset_info["sha256"] = sha256_file(dataset_path)\n\n    requirements_lock = None\n    if not args.skip_pip_freeze:\n        proc = subprocess.run(\n            [sys.executable, "-m", "pip", "freeze"],\n            check=False,\n            text=True,\n            capture_output=True,\n        )\n        if proc.returncode == 0:\n            lock_path = bundle_dir / "requirements.lock.txt"\n            lock_path.write_text(proc.stdout)\n            requirements_lock = str(lock_path.relative_to(bundle_dir))\n            checksums[requirements_lock] = sha256_file(lock_path)\n            copied_files.append(requirements_lock)\n\n    git_commit = None\n    repo_root = Path(__file__).resolve().parents[2]\n    proc = subprocess.run(\n        ["git", "rev-parse", "HEAD"],\n        cwd=repo_root,\n        check=False,\n        text=True,\n        capture_output=True,\n    )\n    if proc.returncode == 0:\n        git_commit = proc.stdout.strip()\n\n    manifest = {\n        "created_utc": utc_now(),\n        "run_name": run_name,\n        "run_dir": str(run_dir),\n        "bundle_dir": str(bundle_dir),\n        "git_commit": git_commit,\n        "host": platform.platform(),\n        "python": platform.python_version(),\n        "script": str(Path(__file__).resolve()),\n        "dataset": dataset_info,\n        "onnx_path": str(args.onnx_path.resolve()) if args.onnx_path else None,\n        "note": args.note,\n        "copied_files": sorted(copied_files),\n    }\n    manifest_path = bundle_dir / "manifest.json"\n    manifest_path.write_text(json.dumps(manifest, indent=2))\n    checksums["manifest.json"] = sha256_file(manifest_path)\n\n    checksum_lines = [f"{digest}  {rel}" for rel, digest in sorted(checksums.items())]\n    (bundle_dir / "checksums.sha256").write_text("\\n".join(checksum_lines) + "\\n")\n\n    zip_path = None\n    if not args.no_zip:\n        zip_path = shutil.make_archive(str(bundle_dir), "zip", root_dir=bundle_dir)\n\n    print(f"Bundle directory: {bundle_dir}")\n    if zip_path:\n        print(f"Bundle zip: {zip_path}")\n    print(f"Copied files: {len(copied_files)}")\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n',
    'scripts/build_submission_variant.py': '#!/usr/bin/env python3\n"""Build a submission zip with tuned run.py thresholds baked in."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport re\nimport tempfile\nimport zipfile\nfrom pathlib import Path\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(description="Build submission zip from run.py + ONNX model.")\n    parser.add_argument("--model-path", required=True, type=Path, help="Path to model.onnx.")\n    parser.add_argument("--output-zip", required=True, type=Path, help="Output submission zip path.")\n    parser.add_argument("--run-template", default=Path("src/inference/run.py"), type=Path)\n    parser.add_argument("--conf", type=float, default=0.03, help="Confidence threshold.")\n    parser.add_argument("--iou", type=float, default=0.60, help="NMS IoU threshold.")\n    parser.add_argument("--topk", type=int, default=2000, help="Top-K candidates before NMS.")\n    parser.add_argument("--max-det", type=int, default=300, help="Max detections per image after NMS.")\n    return parser.parse_args()\n\n\ndef patched_run_py(\n    template_text: str, conf: float, iou: float, topk: int, max_det: int\n) -> str:\n    out = template_text\n    out, n_conf = re.subn(r"conf_thr\\s*=\\s*[0-9]*\\.?[0-9]+", f"conf_thr = {conf:.6f}", out, count=1)\n    out, n_iou = re.subn(r"iou_thr\\s*=\\s*[0-9]*\\.?[0-9]+", f"iou_thr = {iou:.6f}", out, count=1)\n    out, n_max = re.subn(r"max_det\\s*=\\s*\\d+", f"max_det = {max_det}", out, count=1)\n    out, n_topk = re.subn(\n        r"topk\\s*=\\s*min\\(\\s*\\d+\\s*,\\s*conf\\.shape\\[0\\]\\s*\\)",\n        f"topk = min({topk}, conf.shape[0])",\n        out,\n        count=1,\n    )\n\n    if not (n_conf == n_iou == n_max == n_topk == 1):\n        raise RuntimeError(\n            "Could not patch run.py thresholds safely. "\n            f"matches: conf={n_conf}, iou={n_iou}, max_det={n_max}, topk={n_topk}"\n        )\n    return out\n\n\ndef main() -> int:\n    args = parse_args()\n\n    run_template = args.run_template.resolve()\n    model_path = args.model_path.resolve()\n    output_zip = args.output_zip.resolve()\n\n    if not run_template.exists():\n        raise FileNotFoundError(f"run template not found: {run_template}")\n    if not model_path.exists():\n        raise FileNotFoundError(f"model not found: {model_path}")\n\n    run_text = run_template.read_text()\n    run_patched = patched_run_py(\n        template_text=run_text,\n        conf=args.conf,\n        iou=args.iou,\n        topk=args.topk,\n        max_det=args.max_det,\n    )\n\n    output_zip.parent.mkdir(parents=True, exist_ok=True)\n\n    with tempfile.TemporaryDirectory(prefix="ngd_variant_") as td:\n        td_path = Path(td)\n        run_py = td_path / "run.py"\n        model_onnx = td_path / "model.onnx"\n        run_py.write_text(run_patched)\n        model_onnx.write_bytes(model_path.read_bytes())\n\n        with zipfile.ZipFile(output_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:\n            zf.write(run_py, arcname="run.py")\n            zf.write(model_onnx, arcname="model.onnx")\n\n    print(f"Built: {output_zip}")\n    print(\n        f"Params: conf={args.conf:.4f}, iou={args.iou:.4f}, "\n        f"topk={args.topk}, max_det={args.max_det}"\n    )\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n',
    'scripts/validate_submission_local.py': '#!/usr/bin/env python3\n"""Local contract check for submission run.py."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport shutil\nimport subprocess\nimport sys\nimport tempfile\nfrom pathlib import Path\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--input", required=True, type=Path, help="Input directory with test images.")\n    parser.add_argument("--run-path", default=Path("src/inference/run.py"), type=Path)\n    parser.add_argument(\n        "--model-path",\n        default=None,\n        type=Path,\n        help="Optional ONNX model path. If omitted, run.py fallback path is exercised.",\n    )\n    return parser.parse_args()\n\n\ndef main() -> int:\n    args = parse_args()\n    if not args.run_path.exists():\n        raise FileNotFoundError(f"run.py not found: {args.run_path}")\n    if not args.input.exists():\n        raise FileNotFoundError(f"input dir not found: {args.input}")\n\n    with tempfile.TemporaryDirectory(prefix="ngd_validate_") as tmp_dir:\n        workdir = Path(tmp_dir)\n        run_dst = workdir / "run.py"\n        model_dst = workdir / "model.onnx"\n        output_path = workdir / "predictions.local.json"\n\n        shutil.copy2(args.run_path, run_dst)\n        if args.model_path is not None:\n            if not args.model_path.exists():\n                raise FileNotFoundError(f"model not found: {args.model_path}")\n            shutil.copy2(args.model_path, model_dst)\n\n        cmd = [\n            sys.executable,\n            str(run_dst),\n            "--input",\n            str(args.input.resolve()),\n            "--output",\n            str(output_path),\n        ]\n        subprocess.run(cmd, check=True, cwd=workdir)\n\n        payload = json.loads(output_path.read_text())\n        if not isinstance(payload, list):\n            raise RuntimeError("Output must be a JSON array")\n        if payload:\n            required = {"image_id", "category_id", "bbox", "score"}\n            for idx, item in enumerate(payload[:25]):\n                missing = required - set(item)\n                if missing:\n                    raise RuntimeError(f"Missing keys in prediction[{idx}]: {sorted(missing)}")\n                if not isinstance(item["bbox"], list) or len(item["bbox"]) != 4:\n                    raise RuntimeError(f"Invalid bbox format in prediction[{idx}]")\n        print(f"OK: wrote {len(payload)} predictions")\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n',
    'scripts/check_artifact_lineage.py': '#!/usr/bin/env python3\n"""Verify that a run has recoverable checkpoints, audit zip, and submission zip."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport zipfile\nfrom pathlib import Path\nfrom typing import Any\n\n\nMAX_SUBMISSION_BYTES = 420 * 1024 * 1024\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(description="Check run lineage completeness for one run name.")\n    parser.add_argument("--run-name", required=True, help="Run name, e.g. ngd_y11x_1280_e220_l4g4_opt_s42")\n    parser.add_argument(\n        "--drive-root",\n        type=Path,\n        default=Path("/content/drive/MyDrive/NMIAI26/ngd"),\n        help="Root directory containing checkpoints/, audit/, deliverables/.",\n    )\n    parser.add_argument(\n        "--deliverables-root",\n        type=Path,\n        default=None,\n        help="Optional override for deliverables root (defaults to <drive-root>/deliverables).",\n    )\n    parser.add_argument(\n        "--output-json",\n        type=Path,\n        default=None,\n        help="Optional output path for the full report JSON.",\n    )\n    parser.add_argument(\n        "--strict",\n        action="store_true",\n        help="Exit non-zero when critical checks fail.",\n    )\n    return parser.parse_args()\n\n\ndef latest_file(paths: list[Path]) -> Path | None:\n    files = [p for p in paths if p.exists() and p.is_file()]\n    if not files:\n        return None\n    return sorted(files, key=lambda p: p.stat().st_mtime)[-1]\n\n\ndef zip_file_list(path: Path) -> list[str]:\n    with zipfile.ZipFile(path, "r") as zf:\n        return [n for n in zf.namelist() if not n.endswith("/")]\n\n\ndef inspect_submission_zip(path: Path) -> dict[str, Any]:\n    files = zip_file_list(path)\n    basename = {Path(f).name for f in files}\n    has_run = "run.py" in basename\n    has_model = "model.onnx" in basename\n    extras = sorted([f for f in files if Path(f).name not in {"run.py", "model.onnx"}])\n    size_bytes = path.stat().st_size\n    return {\n        "path": str(path),\n        "size_bytes": size_bytes,\n        "size_mb": round(size_bytes / (1024 * 1024), 3),\n        "under_420mb": size_bytes <= MAX_SUBMISSION_BYTES,\n        "contains_run_py": has_run,\n        "contains_model_onnx": has_model,\n        "extra_files": extras,\n    }\n\n\ndef inspect_audit_zip(path: Path) -> dict[str, Any]:\n    files = zip_file_list(path)\n    has_manifest = any(Path(f).name == "manifest.json" for f in files)\n    has_checksums = any(Path(f).name == "checksums.sha256" for f in files)\n    has_best = any(f.endswith("weights/best.pt") for f in files)\n    has_last = any(f.endswith("weights/last.pt") for f in files)\n    has_any_weights = has_best or has_last\n    return {\n        "path": str(path),\n        "contains_manifest": has_manifest,\n        "contains_checksums": has_checksums,\n        "contains_best_pt": has_best,\n        "contains_last_pt": has_last,\n        "contains_any_weights": has_any_weights,\n    }\n\n\ndef main() -> int:\n    args = parse_args()\n    run_name = args.run_name\n    drive_root = args.drive_root.resolve()\n    deliverables_root = (args.deliverables_root or (drive_root / "deliverables")).resolve()\n\n    checkpoint_dir = drive_root / "checkpoints" / run_name\n    audit_dir = drive_root / "audit"\n    deliverables_dir = deliverables_root / run_name\n\n    checkpoint_last = checkpoint_dir / "weights" / "last.pt"\n    checkpoint_best = checkpoint_dir / "weights" / "best.pt"\n    checkpoint_ok = checkpoint_last.exists() or checkpoint_best.exists()\n\n    latest_audit = latest_file(list(audit_dir.glob(f"{run_name}_*.zip")))\n    audit_info = inspect_audit_zip(latest_audit) if latest_audit else None\n\n    selected_submission = deliverables_dir / f"submission_{run_name}_selected.zip"\n    if selected_submission.exists():\n        submission_path = selected_submission\n    else:\n        submission_path = latest_file(list(deliverables_dir.glob("submission*.zip")))\n\n    submission_info = inspect_submission_zip(submission_path) if submission_path else None\n\n    critical = {\n        "checkpoint_exists": checkpoint_ok,\n        "audit_zip_exists": latest_audit is not None,\n        "audit_zip_has_manifest": bool(audit_info and audit_info["contains_manifest"]),\n        "audit_zip_has_checksums": bool(audit_info and audit_info["contains_checksums"]),\n        "audit_zip_has_weights": bool(audit_info and audit_info["contains_any_weights"]),\n        "submission_zip_exists": submission_path is not None,\n        "submission_has_run_py": bool(submission_info and submission_info["contains_run_py"]),\n        "submission_has_model_onnx": bool(submission_info and submission_info["contains_model_onnx"]),\n        "submission_under_420mb": bool(submission_info and submission_info["under_420mb"]),\n    }\n    passed = all(critical.values())\n\n    report = {\n        "run_name": run_name,\n        "drive_root": str(drive_root),\n        "paths": {\n            "checkpoint_dir": str(checkpoint_dir),\n            "audit_dir": str(audit_dir),\n            "deliverables_dir": str(deliverables_dir),\n        },\n        "checkpoints": {\n            "last_pt": str(checkpoint_last),\n            "last_pt_exists": checkpoint_last.exists(),\n            "best_pt": str(checkpoint_best),\n            "best_pt_exists": checkpoint_best.exists(),\n        },\n        "audit": audit_info,\n        "submission": submission_info,\n        "critical_checks": critical,\n        "passed": passed,\n    }\n\n    print(json.dumps(report, indent=2))\n    if args.output_json is not None:\n        args.output_json.parent.mkdir(parents=True, exist_ok=True)\n        args.output_json.write_text(json.dumps(report, indent=2))\n\n    if args.strict and not passed:\n        return 1\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n',
    }
    for rel, content in embedded_files.items():
        dst = WORKDIR / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        dst.write_text(content)
elif source_mode == 'drive':
    src = Path(CFG['DRIVE_CODE_PATH'])
    if not src.exists():
        raise FileNotFoundError(f"Drive code path not found: {src}")
    shutil.copytree(src, WORKDIR, dirs_exist_ok=True, ignore=ignore)
elif source_mode == 'git':
    if not CFG['GIT_URL']:
        raise RuntimeError("Set CFG['GIT_URL'] when CODE_SOURCE='git'.")

    clone_url = CFG['GIT_URL']
    token = str(CFG.get('GIT_TOKEN', '')).strip()
    if not token:
        token = getpass.getpass('GitHub token (leave empty for public repo): ').strip()
    if token:
        clone_url = clone_url.replace('https://', f'https://x-access-token:{token}@')

    subprocess.run(
        ['git', 'clone', '--depth', '1', '--branch', CFG['GIT_REF'], clone_url, str(WORKDIR)],
        check=True,
    )
    del token
elif source_mode == 'upload_zip':
    zip_path_str = str(CFG.get('SOURCE_ZIP_IN_DRIVE', '')).strip()
    if zip_path_str:
        src_zip = Path(zip_path_str)
        if not src_zip.exists():
            raise FileNotFoundError(f"SOURCE_ZIP_IN_DRIVE not found: {src_zip}")
    else:
        from google.colab import files
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError('No source zip uploaded.')
        src_zip = Path(next(iter(uploaded.keys())))

    unpack_dir = Path('/tmp/ngd_src_unpack')
    if unpack_dir.exists():
        shutil.rmtree(unpack_dir)
    unpack_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(src_zip, 'r') as zf:
        zf.extractall(unpack_dir)

    markers = list(unpack_dir.rglob('src/data/prepare_yolo_dataset.py'))
    if not markers:
        raise RuntimeError('Uploaded zip does not contain expected project files.')
    src_root = markers[0].parents[2]
    shutil.copytree(src_root, WORKDIR, dirs_exist_ok=True, ignore=ignore)
else:
    raise ValueError("CODE_SOURCE must be 'embedded', 'upload_zip', 'git' or 'drive'.")

required = [
    'src/data/prepare_yolo_dataset.py',
    'src/train/train_yolov8.py',
    'src/inference/run.py',
    'scripts/watch_checkpoint_sync.py',
    'scripts/archive_training_run.py',
    'scripts/build_submission_variant.py',
    'scripts/validate_submission_local.py',
    'scripts/check_artifact_lineage.py',
]
missing = [rel for rel in required if not (WORKDIR / rel).exists()]
if missing:
    raise RuntimeError(f"Missing required files after bootstrap: {missing}")

print('Source mode:', source_mode)
print('Workdir:', WORKDIR)
print('Files:', os.listdir(WORKDIR)[:20])



In [ ]:
# ---- 4) Stage dataset locally (avoid browser upload bottleneck) ----
import shutil
dataset_src = Path(CFG['DATASET_ZIP_IN_DRIVE'])
dataset_dst = WORKDIR / 'data' / 'raw' / 'NM_NGD_coco_dataset.zip'
dataset_dst.parent.mkdir(parents=True, exist_ok=True)
if not dataset_src.exists():
    raise FileNotFoundError(f'Dataset not found in Drive: {dataset_src}')
if (not dataset_dst.exists()) or dataset_dst.stat().st_size != dataset_src.stat().st_size:
    shutil.copy2(dataset_src, dataset_dst)
print('Dataset ready:', dataset_dst, 'size_MB=', round(dataset_dst.stat().st_size/1024/1024, 2))


In [ ]:
# ---- 5) Prepare YOLO dataset ----
import subprocess
prep_cmd = [
    'python', 'src/data/prepare_yolo_dataset.py',
    '--zip-path', 'data/raw/NM_NGD_coco_dataset.zip',
    '--out-dir', 'yolo_dataset',
    '--val-ratio', str(CFG['VAL_RATIO']),
    '--seed', str(CFG['SEED']),
]
subprocess.run(prep_cmd, cwd=WORKDIR, check=True)
print('Prepared:', WORKDIR / 'yolo_dataset' / 'data.yaml')


In [ ]:
# ---- 6) Start live checkpoint sync (critical) ----
import subprocess
sync_cmd = [
    'python', 'scripts/watch_checkpoint_sync.py',
    '--run-dir', str(RUN_DIR),
    '--backup-dir', str(BACKUP_DIR),
    '--watch',
    '--wait-for-run-dir',
    '--interval', '90',
]
sync_proc = subprocess.Popen(sync_cmd, cwd=WORKDIR)
print('sync pid:', sync_proc.pid)


In [ ]:
# ---- 7) Train ----
import os
import shutil
import subprocess
import time

# Optional restore from Drive checkpoint mirror for long runs / runtime reconnects.
resume_flag = bool(CFG.get('RESUME_FROM_BACKUP', True))
resume_ckpt = BACKUP_DIR / 'weights' / 'last.pt'
model_arg = CFG['MODEL']

if resume_flag and resume_ckpt.exists():
    for rel in [
        'weights/last.pt',
        'weights/best.pt',
        'args.yaml',
        'results.csv',
        'results.png',
    ]:
        src = BACKUP_DIR / rel
        if src.exists() and src.is_file():
            dst = RUN_DIR / rel
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)
    model_arg = str((RUN_DIR / 'weights' / 'last.pt').resolve())
    print('Resuming from:', model_arg)
else:
    print('Starting fresh from:', model_arg)

train_cmd = [
    'python', '-u', 'src/train/train_yolov8.py',
    '--data', 'yolo_dataset/data.yaml',
    '--model', str(model_arg),
    '--epochs', str(CFG['EPOCHS']),
    '--imgsz', str(CFG['IMGSZ']),
    '--batch', str(CFG['BATCH']),
    '--patience', str(CFG['PATIENCE']),
    '--workers', str(CFG['WORKERS']),
    '--seed', str(CFG['SEED']),
    '--mosaic', str(CFG['MOSAIC']),
    '--close-mosaic', str(CFG['CLOSE_MOSAIC']),
    '--device', str(CFG['DEVICE']),
    '--name', CFG['RUN_NAME'],
]
if resume_flag and resume_ckpt.exists():
    train_cmd.append('--resume')

print('Running:', ' '.join(train_cmd))

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
env['PYTHONIOENCODING'] = 'utf-8'

start = time.time()
proc = subprocess.Popen(train_cmd, cwd=WORKDIR, env=env)
heartbeat_sec = 120

while True:
    return_code = proc.poll()
    if return_code is not None:
        break
    time.sleep(heartbeat_sec)
    elapsed_min = (time.time() - start) / 60.0
    print(f'[heartbeat] training still running ({elapsed_min:.1f} min elapsed)')

elapsed_min = (time.time() - start) / 60.0
print(f'Train process exit={return_code}, elapsed={elapsed_min:.1f} min')
if return_code != 0:
    raise RuntimeError(f'Training failed with exit code {return_code}')

print('Training complete:', RUN_DIR)



In [ ]:
# ---- 8) Export ONNX variants ----
from ultralytics import YOLO

best_pt = RUN_DIR / 'weights' / 'best.pt'
if not best_pt.exists():
    raise FileNotFoundError(best_pt)

model = YOLO(str(best_pt))
onnx_outputs = []
for sz in CFG['EXPORT_IMGSZ']:
    out = model.export(format='onnx', imgsz=int(sz), simplify=True, dynamic=False, opset=17)
    out_path = Path(out)
    target = RUN_DIR / 'weights' / f'best_{int(sz)}.onnx'
    if out_path.resolve() != target.resolve():
        target.write_bytes(out_path.read_bytes())
    onnx_outputs.append(target)
print('ONNX exports:', [str(p) for p in onnx_outputs])


In [ ]:
# ---- 9) Build + benchmark submission candidates, then select best safe zip ----
import json
import shutil
import subprocess
import tempfile
import time
import zipfile

weights_dir = RUN_DIR / 'weights'
if not weights_dir.exists():
    raise FileNotFoundError(weights_dir)

val_dir = WORKDIR / 'yolo_dataset' / 'images' / 'val'
val_images = sorted([p for p in val_dir.glob('*') if p.is_file()])
if not val_images:
    raise FileNotFoundError(f'No benchmark images found in {val_dir}')

bench_n = min(int(CFG['BENCHMARK_SAMPLE_IMAGES']), len(val_images))
bench_dir = WORKDIR / 'tmp_bench_input'
if bench_dir.exists():
    shutil.rmtree(bench_dir)
bench_dir.mkdir(parents=True, exist_ok=True)
for p in val_images[:bench_n]:
    shutil.copy2(p, bench_dir / p.name)
print(f'Benchmark input: {bench_n} images from {val_dir}')

contract_input = WORKDIR / 'data' / 'samples' / 'smoke'
if not contract_input.exists():
    contract_input = bench_dir
print('Contract-check input:', contract_input)

results = []
for rank, profile in enumerate(CFG['SUBMISSION_PROFILES'], start=1):
    export_imgsz = int(profile['export_imgsz'])
    onnx_path = weights_dir / f'best_{export_imgsz}.onnx'
    profile_name = str(profile['name'])

    result = {
        'rank': rank,
        'profile': profile,
        'onnx_path': str(onnx_path),
        'zip_path': None,
        'build_ok': False,
        'contract_ok': False,
        'runtime_ok': False,
        'fallback_like': False,
        'num_predictions': 0,
        'elapsed_sec': None,
        'sec_per_image': None,
        'projected_eval_sec': None,
        'safe': False,
        'error': None,
    }

    try:
        if not onnx_path.exists():
            raise FileNotFoundError(f'ONNX missing for export_imgsz={export_imgsz}: {onnx_path}')

        candidate_name = (
            f"candidate_{rank:02d}_{profile_name}_"
            f"conf{int(float(profile['conf'])*1000):03d}_"
            f"iou{int(float(profile['iou'])*100):03d}.zip"
        )
        candidate_zip = DELIVERABLES_DIR / candidate_name

        build_cmd = [
            'python', 'scripts/build_submission_variant.py',
            '--model-path', str(onnx_path),
            '--output-zip', str(candidate_zip),
            '--run-template', 'src/inference/run.py',
            '--conf', str(profile['conf']),
            '--iou', str(profile['iou']),
            '--topk', str(profile['topk']),
            '--max-det', str(profile['max_det']),
        ]
        subprocess.run(build_cmd, cwd=WORKDIR, check=True)
        result['build_ok'] = True
        result['zip_path'] = str(candidate_zip)

        tmp = Path(tempfile.mkdtemp(prefix=f'ngd_candidate_{rank:02d}_'))
        try:
            with zipfile.ZipFile(candidate_zip, 'r') as zf:
                zf.extractall(tmp)

            # Contract check
            subprocess.run(
                [
                    'python', 'scripts/validate_submission_local.py',
                    '--input', str(contract_input),
                    '--run-path', str(tmp / 'run.py'),
                    '--model-path', str(tmp / 'model.onnx'),
                ],
                cwd=WORKDIR,
                check=True,
            )
            result['contract_ok'] = True

            # Runtime benchmark
            pred_out = tmp / 'predictions.bench.json'
            t0 = time.perf_counter()
            subprocess.run(
                [
                    'python', str(tmp / 'run.py'),
                    '--input', str(bench_dir),
                    '--output', str(pred_out),
                ],
                cwd=tmp,
                check=True,
                timeout=int(CFG['BENCHMARK_TIMEOUT_SEC']),
            )
            elapsed = time.perf_counter() - t0
            result['runtime_ok'] = True
            result['elapsed_sec'] = round(elapsed, 4)
            result['sec_per_image'] = round(elapsed / max(1, bench_n), 6)

            preds = json.loads(pred_out.read_text())
            if not isinstance(preds, list):
                raise RuntimeError('Benchmark output is not a JSON list')
            result['num_predictions'] = len(preds)

            # Detect fallback signature from run.py (3 horizontal bands with fixed low scores)
            if len(preds) == 3 * bench_n and bench_n > 0:
                uniq_scores = sorted({round(float(p.get('score', -1.0)), 2) for p in preds})
                if uniq_scores == [0.21, 0.23, 0.25]:
                    result['fallback_like'] = True

            projected = float(result['sec_per_image']) * float(CFG['EST_EVAL_IMAGES'])
            result['projected_eval_sec'] = round(projected, 3)
            budget = float(CFG['EVAL_TIME_LIMIT_SEC']) * float(CFG['SAFETY_FACTOR'])
            result['safe'] = (
                result['build_ok']
                and result['contract_ok']
                and result['runtime_ok']
                and (not result['fallback_like'])
                and projected <= budget
            )
        finally:
            shutil.rmtree(tmp, ignore_errors=True)

    except subprocess.TimeoutExpired:
        result['error'] = f"benchmark timeout > {CFG['BENCHMARK_TIMEOUT_SEC']}s"
    except Exception as e:
        result['error'] = str(e)

    results.append(result)

# Print concise table
print('\nCandidate summary:')
for r in results:
    print(
        f"[{r['rank']:02d}] {r['profile']['name']:<20} "
        f"build={r['build_ok']} contract={r['contract_ok']} runtime={r['runtime_ok']} "
        f"safe={r['safe']} fallback={r['fallback_like']} "
        f"sec/img={r['sec_per_image']} projected={r['projected_eval_sec']} "
        f"preds={r['num_predictions']}"
        + (f" err={r['error']}" if r['error'] else '')
    )

ok = [
    r for r in results
    if r['build_ok'] and r['contract_ok'] and r['runtime_ok'] and not r['fallback_like'] and r['zip_path']
]
safe = [r for r in ok if r['safe']]

if safe:
    # Highest quality = lowest rank because profiles are ordered by quality priority.
    selected = sorted(safe, key=lambda x: x['rank'])[0]
else:
    if not ok:
        raise RuntimeError('No valid candidate zip produced. Check errors above.')
    # Fallback to fastest valid candidate.
    selected = sorted(ok, key=lambda x: x['sec_per_image'])[0]

selected_src = Path(selected['zip_path'])
submission_zip = DELIVERABLES_DIR / f"submission_{CFG['RUN_NAME']}_selected.zip"
shutil.copy2(selected_src, submission_zip)

benchmark_report = DELIVERABLES_DIR / 'candidate_benchmark.json'
benchmark_report.write_text(json.dumps(results, indent=2))

print('\nSelected profile:', selected['profile'])
print('Selected source zip:', selected_src)
print('Final upload zip:', submission_zip)
print('Benchmark report:', benchmark_report)




In [ ]:
# ---- 10) Stop sync + create accountability zip ----
import subprocess
import shutil

if 'sync_proc' in globals() and sync_proc.poll() is None:
    sync_proc.terminate()
    try:
        sync_proc.wait(timeout=15)
    except Exception:
        sync_proc.kill()
print('sync stopped')

best_onnx_candidates = sorted((RUN_DIR / 'weights').glob('best_*.onnx'))
audit_cmd = [
    'python', 'scripts/archive_training_run.py',
    '--run-dir', str(RUN_DIR),
    '--output-dir', str(AUDIT_DIR),
    '--dataset-path', 'data/raw/NM_NGD_coco_dataset.zip',
    '--note', 'optimized colab pipeline run',
]
if best_onnx_candidates:
    audit_cmd += ['--onnx-path', str(best_onnx_candidates[0])]
subprocess.run(audit_cmd, cwd=WORKDIR, check=True)

# pick the latest audit zip for this run and copy to deliverables
run_prefix = f"{CFG['RUN_NAME']}_"
audit_zips = sorted(
    [p for p in AUDIT_DIR.glob(f"{run_prefix}*.zip") if p.is_file()],
    key=lambda p: p.stat().st_mtime,
)
if not audit_zips:
    raise FileNotFoundError(f"No audit zip found in {AUDIT_DIR} with prefix {run_prefix}")
latest_audit_zip = audit_zips[-1]
audit_copy = DELIVERABLES_DIR / f"audit_{latest_audit_zip.name}"
shutil.copy2(latest_audit_zip, audit_copy)

print('Deliverable 1 (submission):', submission_zip)
print('Deliverable 2 (audit):', audit_copy)
print('Deliverables dir:', DELIVERABLES_DIR)


lineage_report = DELIVERABLES_DIR / 'lineage_report.json'
lineage_cmd = [
    'python', 'scripts/check_artifact_lineage.py',
    '--run-name', CFG['RUN_NAME'],
    '--drive-root', str(DRIVE_ROOT),
    '--deliverables-root', str(Path(CFG['DELIVERABLES_DIR_IN_DRIVE'])),
    '--output-json', str(lineage_report),
    '--strict',
]
subprocess.run(lineage_cmd, cwd=WORKDIR, check=True)
print('Lineage report:', lineage_report)



## Deliverables

This notebook writes the final outputs to:
`DELIVERABLES_DIR_IN_DRIVE/<RUN_NAME>/`

1. Final upload zip: `submission_<RUN_NAME>_selected.zip` (contains only `run.py` + `model.onnx`)
2. Audit zip: `audit_<...>.zip` (weights, run metadata, checksums, environment snapshot)
3. Optional diagnostics: `candidate_*.zip` and `candidate_benchmark.json`

Upload only the final `submission_<RUN_NAME>_selected.zip` to the competition site.
4. Lineage report: `lineage_report.json` (checkpoint/audit/submission integrity summary)

